<a href="https://colab.research.google.com/github/shreyaganesh-123/CSA63--THREAT-INTELLIGENCE-AND-NETWORK-SECURITY/blob/main/EXPERIMENTS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

EXP 1


In [ ]:

def caesar_encrypt(text, shift):
    result = ""

    for ch in text:
        if ch.isalpha():
            base = ord('A') if ch.isupper() else ord('a')
            result += chr((ord(ch) - base + shift) % 26 + base)
        else:
            result += ch

    return result


def caesar_decrypt(text, shift):
    return caesar_encrypt(text, -shift)


message = "AttackAtDawn"
shift = 3

enc = caesar_encrypt(message, shift)
dec = caesar_decrypt(enc, shift)

print("Original :", message)
print("Encrypted:", enc)
print("Decrypted:", dec)

Original : AttackAtDawn
Encrypted: DwwdfnDwGdzq
Decrypted: AttackAtDawn


EXP 2


In [ ]:
import hashlib


def file_hash(filename, algo="sha256"):
    h = hashlib.new(algo)

    with open(filename, "rb") as f:
        h.update(f.read())

    return h.hexdigest()


with open("sample.txt", "w") as f:
    f.write("Confidential report v1")

print("Original hash :", file_hash("sample.txt"))

with open("sample.txt", "a") as f:
    f.write(" - tampered!")

print("Modified hash :", file_hash("sample.txt"))
print("Integrity check: FAILED (hash changed) -> file was modified")

Original hash : 5a54470e71678d9dbac7ffc51f2cd2a76cc1e8e0d1096bd698eef42229f687d7
Modified hash : 66aa68df2112cb433631ca8675173a88a7e6b2b8a64a4942d01221ee3e4ce2e2
Integrity check: FAILED (hash changed) -> file was modified


EXP 3


In [ ]:
import socket


def scan_port(host, port):
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.settimeout(0.5)

    result = s.connect_ex((host, port))

    s.close()

    return result == 0


target = "127.0.0.1"  # Scan your own machine (localhost)

ports = [21, 22, 23, 25, 80, 443, 3306, 8080]

print(f"Scanning {target} ...")

for port in ports:
    status = "OPEN" if scan_port(target, port) else "closed"
    print(f"Port {port}: {status}")

Scanning 127.0.0.1 ...
Port 21: closed
Port 22: closed
Port 23: closed
Port 25: closed
Port 80: closed
Port 443: closed
Port 3306: closed
Port 8080: OPEN


EXP 4

In [ ]:
import re
def check_url(url):
    reasons = []
    if re.match(r"https?://\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}", url):
        reasons.append("Uses raw IP address instead of domain name")
    if "@" in url:
        reasons.append("Contains '@' symbol (can hide real destination)")
    if url.count("-") > 3:
        reasons.append("Too many hyphens in domain (common phishing trick)")
    if any(k in url.lower() for k in ["login", "verify", "update", "secure"]) and "https" not in url:
        reasons.append("Suspicious keyword without HTTPS")
    return reasons
urls = [
    "https://www.google.com",
    "http://192.168.10.5/login",
    "http://paypal-verify-account.com@evil.com",
]
for u in urls:
    issues = check_url(u)
    verdict = "SUSPICIOUS" if issues else "Looks OK"
    print(f"\nURL: {u}\nVerdict: {verdict}")
    for r in issues:
        print(" -", r)


URL: https://www.google.com
Verdict: Looks OK

URL: http://192.168.10.5/login
Verdict: SUSPICIOUS
 - Uses raw IP address instead of domain name
 - Suspicious keyword without HTTPS

URL: http://paypal-verify-account.com@evil.com
Verdict: SUSPICIOUS
 - Contains '@' symbol (can hide real destination)
 - Suspicious keyword without HTTPS


EXP 5

In [ ]:
import socket

domains = [
    "www.google.com",
    "www.python.org",
    "notarealdomain12345.com"
]

for d in domains:
    try:
        ip = socket.gethostbyname(d)
        print(f"{d:30s} -> {ip}")
    except socket.gaierror:
        print(f"{d:30s} -> Could not resolve (invalid/unreachable)")

www.google.com                 -> 142.251.157.119
www.python.org                 -> 151.101.0.223
notarealdomain12345.com        -> Could not resolve (invalid/unreachable)


EXP 6

In [ ]:
# Step 1: Create the sample files (so this script runs standalone,
# with no file upload needed - works on any online compiler)
with open("ioc_list.txt", "w") as f:
    f.write("45.33.32.156\n198.51.100.23\n203.0.113.99\n")
with open("sample_traffic.log", "w") as f:
    f.write("2026-07-08 10:01:12 connection from 10.0.0.5\n")
    f.write("2026-07-08 10:02:44 connection from 45.33.32.156\n")
    f.write("2026-07-08 10:03:10 connection from 192.168.1.20\n")
    f.write("2026-07-08 10:04:55 connection from 203.0.113.99\n")
# Step 2: Load IoCs and scan the log
def load_iocs(filename):
    with open(filename) as f:
        return set(line.strip() for line in f if line.strip())
def scan_log(logfile, iocs):
    with open(logfile) as f:
        for line in f:
            for ip in iocs:
                if ip in line:
                    print("ALERT: Known malicious IP found ->", line.strip())

iocs = load_iocs("ioc_list.txt")

print("Loaded", len(iocs), "IoCs")

scan_log("sample_traffic.log", iocs)

Loaded 3 IoCs
ALERT: Known malicious IP found -> 2026-07-08 10:02:44 connection from 45.33.32.156
ALERT: Known malicious IP found -> 2026-07-08 10:04:55 connection from 203.0.113.99


EXP 7

In [ ]:
import re
from collections import Counter
# Create the sample log so the script runs standalone (no upload needed)
with open("login_attempts.log", "w") as f:
    f.write("2026-07-08 09:00:01 LOGIN SUCCESS user=alice ip=10.0.0.5\n")
    f.write("2026-07-08 09:01:15 LOGIN FAILED user=admin ip=203.0.113.99\n")
    f.write("2026-07-08 09:01:20 LOGIN FAILED user=admin ip=203.0.113.99\n")
    f.write("2026-07-08 09:01:25 LOGIN FAILED user=admin ip=203.0.113.99\n")
    f.write("2026-07-08 09:01:30 LOGIN FAILED user=admin ip=203.0.113.99\n")
    f.write("2026-07-08 09:01:35 LOGIN FAILED user=admin ip=203.0.113.99\n")
    f.write("2026-07-08 09:02:00 LOGIN SUCCESS user=bob ip=10.0.0.8\n")

failed_by_ip = Counter()
with open("login_attempts.log") as f:
    for line in f:
        if "LOGIN FAILED" in line:
            match = re.search(r"ip=(\S+)", line)
            if match:
                failed_by_ip[match.group(1)] += 1
print("Failed login attempts by IP:")
for ip, count in failed_by_ip.items():
    print(f" {ip}: {count} attempts")
    if count >= 5:
        print(f" -> ALERT: possible brute-force attack from {ip}")

Failed login attempts by IP:
 203.0.113.99: 5 attempts
 -> ALERT: possible brute-force attack from 203.0.113.99
